In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install -U "datasets>=3.0.0" tqdm

import ast
import hashlib
import json
import math
import os
import random
import torch.nn.functional as F
from collections import defaultdict
from functools import lru_cache
from itertools import combinations, product

import numpy as np
from datasets import load_dataset

ROOT = "/content/drive/MyDrive/Test"
NS = (2, 3, 4)
TRUTH_OPS = {"k", "t", "truth", "knight", "telling-truth", "truth-teller"}
LIE_OPS = {"n", "l", "lie", "liar", "knave", "lying", "not-knight"}
IFF_OPS = {"<->", "<=>"}
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def mkdirs(*paths):
    for p in paths:
        os.makedirs(p, exist_ok=True)


def run_dirs(name, result_name):
    a = os.path.join(ROOT, name)
    b = os.path.join(a, result_name)
    mkdirs(a, b)
    return a, b


def bstr(x):
    return "True" if bool(x) else "False"


def lit(x):
    return ast.literal_eval(x) if isinstance(x, str) else x


def tree(x):
    return tuple(tree(y) for y in x) if isinstance(x, (list, tuple)) else x


def op(x):
    return x.strip().lower().replace("_", "-").replace(" ", "-") if isinstance(x, str) else None


def seed(x):
    return int(hashlib.md5(x.encode("utf-8")).hexdigest()[:8], 16)


def uid(x):
    s = x if isinstance(x, str) else repr(x)
    return hashlib.md5(s.encode("utf-8")).hexdigest()


def vars_for(n):
    return tuple(chr(ord("a") + i) for i in range(n))


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def split_n(x):
    return int(x[:-3])


def dist(a, b):
    return sum(x != y for x, y in zip(a, b))


@lru_cache(maxsize=None)
def assignments(n):
    return tuple(tuple(bool((bits >> i) & 1) for i in range(n)) for bits in range(1 << n))


def choice(items, weights, rng):
    return rng.choice(items) if not any(weights) else rng.choices(items, weights=weights, k=1)[0]


def pos_rate(rows):
    return float(np.mean([int(r["label"]) for r in rows])) if rows else 0.0


def rname(i):
    return f"rule_{i + 1}"


def neg_atom(x):
    if x[0] == "const":
        return ("const", not x[1])
    if x[0] == "var":
        return ("not", x[1])
    if x[0] == "not":
        return ("var", x[1])
    raise ValueError(x)


def parse_atom(x):
    x = tree(x)

    if isinstance(x, bool):
        return ("const", bool(x))
    if isinstance(x, int):
        return ("var", x)

    tag = op(x[0])
    arg = x[1]

    if tag in TRUTH_OPS:
        return ("var", arg)
    if tag in LIE_OPS:
        return ("not", arg)
    if tag == "not":
        return neg_atom(parse_atom(arg))

    raise ValueError(x)


def parse_expr(x):
    x = tree(x)

    if isinstance(x, (bool, int)) or len(x) == 2:
        return ("atom", parse_atom(x))

    tag = op(x[0])
    a, b = x[1], x[2]

    if tag == "and":
        return ("and", parse_atom(a), parse_atom(b))
    if tag == "or":
        return ("or", parse_atom(a), parse_atom(b))
    if tag in IFF_OPS:
        return ("iff", parse_atom(a), parse_atom(b))
    if tag == "->":
        return ("or", neg_atom(parse_atom(a)), parse_atom(b))

    raise ValueError(x)


def eval_atom(x, a):
    if x[0] == "const":
        return bool(x[1])
    if x[0] == "var":
        return bool(a[x[1]])
    if x[0] == "not":
        return not bool(a[x[1]])
    raise ValueError(x)


def eval_expr(x, a):
    if x[0] == "atom":
        return eval_atom(x[1], a)
    if x[0] == "and":
        return eval_atom(x[1], a) and eval_atom(x[2], a)
    if x[0] == "or":
        return eval_atom(x[1], a) or eval_atom(x[2], a)
    if x[0] == "iff":
        return eval_atom(x[1], a) == eval_atom(x[2], a)
    raise ValueError(x)


def score(stmts, a):
    return sum(a[i] == eval_expr(stmts[i], a) for i in range(len(stmts)))


def holds(stmts, a):
    return all(a[i] == eval_expr(stmts[i], a) for i in range(len(stmts)))


def atom_text(x, names, lang):
    if x[0] == "const":
        return bstr(x[1])

    name = names[x[1]]

    if x[0] == "var":
        return name
    if x[0] == "not" and lang == "py":
        return f"(not {name})"
    if x[0] == "not" and lang == "en":
        return f"not {name}"

    raise ValueError((x, lang))


def expr_text(x, names, lang):
    if x[0] == "atom":
        return atom_text(x[1], names, lang)

    left = atom_text(x[1], names, lang)
    right = atom_text(x[2], names, lang)

    if lang == "py":
        if x[0] == "and":
            return f"{left} and {right}"
        if x[0] == "or":
            return f"{left} or {right}"
        if x[0] == "iff":
            return f"{left} == {right}"

    if lang == "en":
        if x[0] == "and":
            return f"both {left} and {right}"
        if x[0] == "or":
            return f"either {left} or {right}"
        if x[0] == "iff":
            return f"{left} if and only if {right}"

    raise ValueError((x, lang))


def py_prompt(stmts, a, names):
    lines = [
        "Instruction:",
        "Evaluate the rules on the given input.",
        "Output only: True or False.",
        "",
        "Rules:",
    ]

    for i, x in enumerate(stmts):
        lines.append(f"{rname(i)} = {expr_text(x, names, 'py')}")

    lines += [
        "",
        "    return " + " and ".join(f"{name} == {rname(i)}" for i, name in enumerate(names)),
        "",
        "Input:",
    ]

    for name, val in zip(names, a):
        lines.append(f"{name} = {bstr(val)}")

    lines += ["", "Output:"]
    return "\n".join(lines)


def en_prompt(stmts, a, names):
    lines = [
        "Instruction:",
        "Evaluate the rules on the given input.",
        "Output only: True or False.",
        "",
        "Rules:",
    ]

    for i, x in enumerate(stmts):
        lines.append(f"{rname(i)} is defined as: {expr_text(x, names, 'en')}")

    checks = [f"{name} is True if and only if {rname(i)} is True" for i, name in enumerate(names)]

    lines += [
        "",
        "The output states whether: " + " and ".join(checks) + ".",
        "",
        "Input:",
    ]

    for name, val in zip(names, a):
        lines.append(f"{name} is defined as: {bstr(val)}")

    lines += ["", "Output:"]
    return "\n".join(lines)


In [ ]:
DATASET = "K-and-K/knights-and-knaves"
PREP_ROOT, PREP_RES = run_dirs("PreparedData", "prepared_data_results")

VAL_FRAC = 0.10
FALSE_FACTOR = 1.0
MIN_TRUE = 4
MAX_TRUE = 24
N_MULT = {2: 0.90, 3: 1.15, 4: 1.50}
IFF_MULT = 1.25

PHASES = [
    {"name": "main", "title": "Main balanced phase (English only)", "epochs": 1, "false_per_true": 1, "scale": 1.0, "filter": None},
    {"name": "finish", "title": "Short false-heavy finishing phase (English only)", "epochs": 1, "false_per_true": 2, "scale": 0.10, "filter": lambda p: p["n"] == 4 or p["has_iff"]},
]

P_TRUE = 0.65
P_FALSE = 0.70
D_BIAS = 0.20
S_BIAS = 0.15


def split_solutions(stmts):
    all_a = assignments(len(stmts))
    sol = [a for a in all_a if holds(stmts, a)]
    sol_set = set(sol)
    bad = [a for a in all_a if a not in sol_set]
    return sol, sol_set, bad


def load_puzzles(split):
    out = {}
    data = load_dataset(DATASET, split)

    for part, rows in data.items():
        n = split_n(part)
        if n not in NS:
            continue

        for row in rows:
            stmts = tuple(parse_expr(x) for x in tree(lit(row["statements"])))
            sol, sol_set, bad = split_solutions(stmts)

            if len(sol) == 0 or len(sol) == (1 << n):
                continue

            k = uid(stmts)
            if k in out:
                continue

            out[k] = {
                "uid": k,
                "n": n,
                "names": vars_for(n),
                "stmts": stmts,
                "sol": sol,
                "sol_set": sol_set,
                "bad": bad,
                "has_iff": any(x[0] == "iff" for x in stmts),
                "bad_score": {a: score(stmts, a) for a in bad},
            }

    return list(out.values())


def example(p, a, split):
    y = a in p["sol_set"]
    return {
        "uid": p["uid"],
        "split": split,
        "n": p["n"],
        "assignment": {name: bool(val) for name, val in zip(p["names"], a)},
        "label": bool(y),
        "english_text": en_prompt(p["stmts"], a, p["names"]),
        "python_text": py_prompt(p["stmts"], a, p["names"]),
    }


def split_train_val(items):
    groups = defaultdict(list)
    for p in items:
        groups[(p["n"], int(p["has_iff"]))].append(p)

    train, val = [], []
    for key, group in groups.items():
        group = sorted(group, key=lambda p: p["uid"])
        rng = random.Random(seed(f"val-split-{key}"))
        rng.shuffle(group)
        k = 0 if len(group) <= 1 else min(len(group) - 1, max(1, int(round(len(group) * VAL_FRAC))))
        val += group[:k]
        train += group[k:]

    return train, val


def true_draws(false_pool, p, scale):
    x = max(MIN_TRUE, int(math.ceil(FALSE_FACTOR * len(false_pool))))
    x = min(x, MAX_TRUE)

    m = N_MULT.get(p["n"], 1.0)
    if p["has_iff"]:
        m *= IFF_MULT

    x = max(1, int(round(x * m * scale)))
    return min(x, max(1, int(math.ceil(MAX_TRUE * max(1.0, m * scale)))))


def use_pool(pool, used, rng, prob, last):
    unseen = [a for a in pool if used[a] == 0]
    seen = [a for a in pool if used[a] > 0]
    if unseen and seen:
        cand = unseen if rng.random() < prob else seen
    else:
        cand = unseen or seen or pool

    if last is not None and len(cand) > 1:
        cand = [a for a in cand if a != last] or cand

    return cand


def pick_true(pool, used, rng, last):
    cand = use_pool(pool, used, rng, P_TRUE, last)
    a = choice(cand, [1.0 / (1.0 + used[x]) for x in cand], rng)
    used[a] += 1
    return a


def pick_false(pool, used, bucket_used, good, rng, last, p):
    cand = use_pool(pool, used, rng, P_FALSE, last)
    weights = []

    for a in cand:
        d = dist(a, good)
        u = 1.0 / (1.0 + used[a])
        bucket = 1.0 / (1.0 + bucket_used[d])
        close = 1.0 + D_BIAS * ((p["n"] - d) / max(1, p["n"]))
        plaus = p["bad_score"][a] / max(1, p["n"])
        weights.append(u * bucket * close * (1.0 + S_BIAS * plaus))

    a = choice(cand, weights, rng)
    d = dist(a, good)
    used[a] += 1
    bucket_used[d] += 1
    return a, d


def make_phase(items, split, false_per_true, scale, keep=None):
    rows, cov = [], []

    for p in items:
        if keep is not None and not keep(p):
            continue

        good_pool = list(p["sol"])
        bad_pool = list(p["bad"])
        n_good = true_draws(bad_pool, p, scale)

        rng_good = random.Random(seed(f"{split}-true-{p['uid']}"))
        rng_bad = random.Random(seed(f"{split}-false-{p['uid']}"))

        good_used = {a: 0 for a in good_pool}
        bad_used = {a: 0 for a in bad_pool}
        bucket_used = defaultdict(int)
        hist = defaultdict(int)

        last_good = None
        last_bad = None

        for _ in range(n_good):
            good = pick_true(good_pool, good_used, rng_good, last_good)
            last_good = good
            rows.append(example(p, good, split))

            for _ in range(false_per_true):
                bad, d = pick_false(bad_pool, bad_used, bucket_used, good, rng_bad, last_bad, p)
                last_bad = bad
                hist[d] += 1
                rows.append(example(p, bad, split))

        cov.append({
            "uid": p["uid"],
            "split": split,
            "n": p["n"],
            "has_iff": bool(p["has_iff"]),
            "false_per_true": int(false_per_true),
            "phase_scale": float(scale),
            "num_true_available": len(good_pool),
            "num_false_available": len(bad_pool),
            "true_draws": int(n_good),
            "num_true_used_total": int(sum(good_used.values())),
            "num_false_used_total": int(sum(bad_used.values())),
            "num_true_used_unique": int(sum(v > 0 for v in good_used.values())),
            "num_false_used_unique": int(sum(v > 0 for v in bad_used.values())),
            "true_full_coverage_reached": sum(v > 0 for v in good_used.values()) == len(good_pool),
            "false_full_coverage_reached": sum(v > 0 for v in bad_used.values()) == len(bad_pool),
            "true_usage_counts": {str(k): int(v) for k, v in sorted(good_used.items(), key=lambda kv: kv[0])},
            "false_usage_counts": {str(k): int(v) for k, v in sorted(bad_used.items(), key=lambda kv: kv[0])},
            "false_distance_hist": {str(k): int(v) for k, v in sorted(hist.items())},
        })

    random.Random(seed("shuffle-" + split)).shuffle(rows)
    return rows, cov


def make_train(items):
    out = {}
    for p in PHASES:
        rows, cov = make_phase(items, f"train_{p['name']}", p["false_per_true"], p["scale"], p["filter"])
        out[p["name"]] = {"rows": rows, "coverage": cov}
    return out


def make_full(items, split):
    return [example(p, a, split) for p in items for a in assignments(p["n"])]


def full_stats(items):
    total = sum(1 << p["n"] for p in items)
    pos = sum(len(p["sol"]) for p in items)
    return {"num_examples": int(total), "positive_rate": float(pos / total) if total else 0.0}


def pick_text(rows, key):
    return [{"text": r[key], "label": r["label"], "uid": r["uid"], "assignment": r["assignment"], "n": r["n"], "split": r["split"]} for r in rows]


def cov_stats(rows):
    def avg(k):
        return float(np.mean([r[k] for r in rows])) if rows else 0.0

    return {
        "num_puzzles": len(rows),
        "avg_true_draws": avg("true_draws"),
        "avg_false_per_true": avg("false_per_true"),
        "avg_true_available": avg("num_true_available"),
        "avg_false_available": avg("num_false_available"),
        "avg_true_unique_used": avg("num_true_used_unique"),
        "avg_false_unique_used": avg("num_false_used_unique"),
        "all_true_full_coverage": bool(all(r["true_full_coverage_reached"] for r in rows)) if rows else True,
        "all_false_full_coverage": bool(all(r["false_full_coverage_reached"] for r in rows)) if rows else True,
    }


def py_style(names, body, a):
    lines = [
        "Instruction:",
        "Evaluate the rules on the given input.",
        "Output only: True or False.",
        "",
        "Rules:",
    ]

    for x in body:
        if x == "":
            lines.append("")
        elif x.startswith("check = "):
            lines.append("    return " + x[len("check = "):])
        elif x.startswith("return "):
            lines.append("    " + x)
        else:
            lines.append(x)

    lines += ["", "Input:"]

    for name, val in zip(names, a):
        lines.append(f"{name} = {bstr(val)}")

    lines += ["", "Output:"]
    return "\n".join(lines)


def add(rows, fam, n, names, body, a, y, meta=None):
    text = py_style(names, body, a)
    row = {
        "uid": uid(text + "|||" + bstr(y)),
        "family": fam,
        "n": n,
        "assignment": {name: bool(val) for name, val in zip(names, a)},
        "text": text,
        "label": bool(y),
    }
    if meta:
        row.update(meta)
    rows.append(row)


def add_all(rows, fam, n, names, body, fn, meta=None):
    for a in assignments(n):
        add(rows, fam, n, names, body, a, fn(a), meta)


def dedup(rows):
    seen, out = set(), []
    for r in rows:
        k = (r["text"], bool(r["label"]))
        if k not in seen:
            seen.add(k)
            out.append(r)
    return out


def signed(n):
    for i in range(n):
        yield ("var", i)
        yield ("not", i)


def signed_name(x):
    return f"{x[0]}_{x[1]}"


def signed_meta(prefix, x):
    return {f"{prefix}_kind": x[0], f"{prefix}_var": x[1], f"{prefix}_signed_atom": repr(x)}


def exprs(n):
    for x in signed(n):
        suffix = "var" if x[0] == "var" else "not"
        e = ("atom", x)
        yield suffix, e, {
            "expr": repr(e),
            "source_kind": x[0],
            "source_var": x[1],
            "source_signed_atom": repr(x),
            "expr_arity": 1,
            "expr_tag": "atom",
        }

    for a, b in product(signed(n), repeat=2):
        for label, tag in (("and", "and"), ("or", "or"), ("eq", "iff")):
            e = (tag, a, b)
            yield f"{label}_{signed_name(a)}__{signed_name(b)}", e, {
                "expr": repr(e),
                "expr_arity": 2,
                "expr_tag": tag,
                **signed_meta("left", a),
                **signed_meta("right", b),
            }


def add_rule(rows, n, names, fam, target, e, meta):
    rule = rname(target)
    body = [f"{rule} = {expr_text(e, names, 'py')}", "", f"check = ({names[target]} == {rule})"]
    info = {"target_rule": rule, "target_idx": target, "family_mode": "local_check", **meta}
    info.setdefault("rule_expr", repr(e))
    add_all(rows, fam, n, names, body, lambda a, target=target, e=e: bool(a[target]) == bool(eval_expr(e, a)), info)


def add_multi(rows, n, names, fam, targets, sources):
    body = [f"{rname(i)} = {names[j]}" for i, j in enumerate(sources)]
    body += ["", "check = " + " and ".join(f"({names[t]} == {rname(i)})" for i, t in enumerate(targets))]

    add_all(
        rows,
        fam,
        n,
        names,
        body,
        lambda a, targets=targets, sources=sources: all(bool(a[t]) == bool(a[s]) for t, s in zip(targets, sources)),
        {
            "target_vars": [names[i] for i in targets],
            "source_vars": [names[i] for i in sources],
            "num_checks": len(targets),
            "has_repeated_source": len(set(sources)) < len(sources),
        },
    )


def atom_rows(n):
    names = vars_for(n)
    specs = list(exprs(n))
    rows = []

    for target in range(n):
        for suffix, e, meta in specs:
            add_rule(rows, n, names, f"local_check_{suffix}", target, e, meta)

    for k, fam in ((2, "double_local_check"), (3, "triple_local_check"), (4, "quad_local_check")):
        if n >= k:
            for targets in combinations(range(n), k):
                for sources in product(range(n), repeat=k):
                    add_multi(rows, n, names, fam, targets, sources)

    rows = dedup(rows)
    rows.sort(key=lambda r: (r["n"], r["family"], r["text"], int(r["label"])))
    return rows


def build_data():
    train = load_puzzles("train")
    test = load_puzzles("test")
    train, val = split_train_val(train)

    train_ids = {p["uid"] for p in train}
    val_ids = {p["uid"] for p in val}
    test_ids = {p["uid"] for p in test}

    overlap_before = {
        "train_val": len(train_ids & val_ids),
        "train_test": len(train_ids & test_ids),
        "val_test": len(val_ids & test_ids),
    }
    print("[prepared] split UID overlaps before cleanup:", overlap_before)

    clean_test_ids = test_ids
    clean_val_ids = val_ids - test_ids
    clean_train_ids = train_ids - val_ids - test_ids

    skipped = {
        "train": len(train_ids) - len(clean_train_ids),
        "val": len(val_ids) - len(clean_val_ids),
        "test": 0,
    }
    print("[prepared] skipped overlapping UIDs:", skipped)

    train = [p for p in train if p["uid"] in clean_train_ids]
    val = [p for p in val if p["uid"] in clean_val_ids]
    test = [p for p in test if p["uid"] in clean_test_ids]

    train_ids = {p["uid"] for p in train}
    val_ids = {p["uid"] for p in val}
    test_ids = {p["uid"] for p in test}

    overlap = {
        "train_val": len(train_ids & val_ids),
        "train_test": len(train_ids & test_ids),
        "val_test": len(val_ids & test_ids),
    }
    print("[prepared] split UID overlaps after cleanup:", overlap)

    phase = make_train(train)
    val_rows = make_full(val, "val")
    test_rows = make_full(test, "test")
    train_stats = full_stats(train)

    en_train_paths = {p["name"]: os.path.join(PREP_ROOT, f"english_train_{p['name']}.jsonl") for p in PHASES}
    for name, path in en_train_paths.items():
        save_jsonl(path, pick_text(phase[name]["rows"], "english_text"))

    en_val = os.path.join(PREP_ROOT, "english_val.jsonl")
    en_test = os.path.join(PREP_ROOT, "english_test.jsonl")
    py_val = os.path.join(PREP_ROOT, "python_val.jsonl")
    py_test = os.path.join(PREP_ROOT, "python_test.jsonl")
    atoms_path = os.path.join(PREP_ROOT, "python_strict_atoms_train.jsonl")

    save_jsonl(en_val, pick_text(val_rows, "english_text"))
    save_jsonl(en_test, pick_text(test_rows, "english_text"))
    save_jsonl(py_val, pick_text(val_rows, "python_text"))
    save_jsonl(py_test, pick_text(test_rows, "python_text"))

    atoms = [r for n in NS for r in atom_rows(n)]
    save_jsonl(atoms_path, atoms)

    coverage = {name: cov_stats(phase[name]["coverage"]) for name in phase}
    save_json(os.path.join(PREP_RES, "english_phase_coverage_summary.json"), coverage)
    save_json(os.path.join(PREP_RES, "english_phase_coverage_raw.json"), {name: phase[name]["coverage"] for name in phase})

    overview = {
        "dataset_id": DATASET,
        "target_ns": list(NS),
        "split_protection": {
            "unit": "puzzle uid",
            "train_puzzles": len(train_ids),
            "val_puzzles": len(val_ids),
            "test_puzzles": len(test_ids),
            "overlaps": overlap,
        },
        "counts": {
            "train_examples_full": train_stats["num_examples"],
            "english_train_main": len(phase["main"]["rows"]),
            "english_train_finish": len(phase["finish"]["rows"]),
            "english_val": len(val_rows),
            "english_test": len(test_rows),
            "python_val": len(val_rows),
            "python_test": len(test_rows),
            "python_strict_atoms_train": len(atoms),
        },
        "positive_rates": {
            "train_full": train_stats["positive_rate"],
            "english_train_main": pos_rate(phase["main"]["rows"]),
            "english_train_finish": pos_rate(phase["finish"]["rows"]),
            "english_val": pos_rate(val_rows),
            "english_test": pos_rate(test_rows),
            "python_strict_atoms_train": pos_rate(atoms),
        },
        "artifacts": {
            "english_train_main": en_train_paths["main"],
            "english_train_finish": en_train_paths["finish"],
            "english_val": en_val,
            "english_test": en_test,
            "python_val": py_val,
            "python_test": py_test,
            "python_strict_atoms_train": atoms_path,
            "coverage_summary": os.path.join(PREP_RES, "english_phase_coverage_summary.json"),
            "coverage_raw": os.path.join(PREP_RES, "english_phase_coverage_raw.json"),
        },
    }

    save_json(os.path.join(PREP_RES, "prepared_data_overview.json"), overview)

    print("\nSaved prepared data to:")
    print(PREP_ROOT)
    print("\nOverview:")
    print(json.dumps(overview, indent=2, ensure_ascii=False))


build_data()


In [ ]:
!pip -q install -U "transformers>=4.46.0" "peft>=0.13.0" "accelerate>=1.0.0" "torchao>=0.16.0" safetensors tqdm

import gc
import torch
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

MODEL = "EleutherAI/pythia-1.4b-deduped"

PREP_ROOT = os.path.join(ROOT, "PreparedData")
OUT = os.path.join(ROOT, "Python", "python_atoms_aux")
RES = os.path.join(OUT, "results")
ALL_PATH = os.path.join(PREP_ROOT, "python_strict_atoms_train.jsonl")
TRAIN_PATH = os.path.join(RES, "python_strict_atoms_train_split.jsonl")
VAL_PATH = os.path.join(RES, "python_strict_atoms_val_split.jsonl")
BEST_DIR = os.path.join(OUT, "best")
LAST_DIR = os.path.join(OUT, "last")
mkdirs(RES, BEST_DIR, LAST_DIR)

SEED = 42
DEVICE = "cuda"
DTYPE = torch.float16

EPOCHS = 10
TRAIN_BS = 16
EVAL_BS = 16
ACCUM = 1
LR = 5e-5
MAX_LEN = 256
WD = 0.01
WARMUP = 0.05
VAL_FRAC = 0.10

R = 64
ALPHA = 64
DROPOUT = 0.10
MODULES = ["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"]

os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]


def clear():
    gc.collect()
    torch.cuda.empty_cache()


class SFT(Dataset):
    def __init__(self, rows, tok):
        self.items = []
        for r in rows:
            ids = tok(r["text"], add_special_tokens=False).input_ids[:MAX_LEN]
            self.items.append({"input_ids": ids, "label": int(r["label"])})

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def collate(batch, pad_id):
    n = max(len(x["input_ids"]) for x in batch)
    ids, mask, labels = [], [], []

    for x in batch:
        k = n - len(x["input_ids"])
        ids.append(x["input_ids"] + [pad_id] * k)
        mask.append([1] * len(x["input_ids"]) + [0] * k)
        labels.append(x["label"])

    return {
        "input_ids": torch.tensor(ids, dtype=torch.long),
        "attention_mask": torch.tensor(mask, dtype=torch.long),
        "label": torch.tensor(labels, dtype=torch.long),
    }


@torch.no_grad()
def eval_on(model, loader):
    model.eval()
    total = 0.0
    correct = 0
    count = 0

    for batch in tqdm(loader, desc="atoms val", leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop("label")

        with torch.amp.autocast("cuda", dtype=DTYPE):
            z = model(**batch).logits
            last = batch["attention_mask"].sum(1) - 1
            z = z[torch.arange(z.size(0), device=DEVICE), last][:, ans_ids].float()
            loss = F.cross_entropy(z, labels)

        pred = z.argmax(dim=1)
        total += loss.item()
        correct += int((pred == labels).sum().item())
        count += int(labels.numel())

    return total / max(1, len(loader)), correct / max(1, count)


tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token

neg_len = len(tok(" False", add_special_tokens=False).input_ids)
pos_len = len(tok(" True", add_special_tokens=False).input_ids)
print(f"[atoms] candidate token lengths | neg={neg_len} | pos={pos_len}")

false_id = tok(" False", add_special_tokens=False).input_ids[0]
true_id = tok(" True", add_special_tokens=False).input_ids[0]
ans_ids = torch.tensor([false_id, true_id], device=DEVICE)

rows = load_jsonl(ALL_PATH)
random.Random(SEED).shuffle(rows)

k = max(1, int(round(len(rows) * VAL_FRAC)))
val_rows = rows[:k]
train_rows = rows[k:]

train_ids = {r["uid"] for r in train_rows}
val_ids = {r["uid"] for r in val_rows}
overlap = train_ids & val_ids

print(f"[atoms] uid overlap train/val before cleanup: {len(overlap)}")

if overlap:
    train_rows = [r for r in train_rows if r["uid"] not in overlap]

train_ids = {r["uid"] for r in train_rows}
val_ids = {r["uid"] for r in val_rows}
overlap = train_ids & val_ids

print(f"[atoms] uid overlap train/val after cleanup: {len(overlap)}")

save_jsonl(TRAIN_PATH, train_rows)
save_jsonl(VAL_PATH, val_rows)

train_data = SFT(train_rows, tok)
val_data = SFT(val_rows, tok)

train_loader = DataLoader(train_data, batch_size=TRAIN_BS, shuffle=True, collate_fn=lambda x: collate(x, tok.pad_token_id))
val_loader = DataLoader(val_data, batch_size=EVAL_BS, shuffle=False, collate_fn=lambda x: collate(x, tok.pad_token_id))

print(f"[atoms] total rows: {len(rows)}")
print(f"[atoms] train rows: {len(train_rows)}")
print(f"[atoms] val rows:   {len(val_rows)}")
print(f"[atoms] uid overlap train/val: {len(overlap)}")

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=R,
    lora_alpha=ALPHA,
    lora_dropout=DROPOUT,
    bias="none",
    target_modules=MODULES,
)

model = get_peft_model(model, cfg)
model.to(DEVICE)
model.print_trainable_parameters()

opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR, weight_decay=WD)
steps = max(1, math.ceil(len(train_loader) / ACCUM) * EPOCHS)
sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(steps * WARMUP), num_training_steps=steps)


best = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    opt.zero_grad(set_to_none=True)

    total = 0.0
    correct = 0
    count = 0
    bar = tqdm(train_loader, desc=f"atoms epoch {epoch}/{EPOCHS}")

    for step, batch in enumerate(bar, start=1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop("label")

        with torch.amp.autocast("cuda", dtype=DTYPE):
            z = model(**batch).logits
            last = batch["attention_mask"].sum(1) - 1
            z = z[torch.arange(z.size(0), device=DEVICE), last][:, ans_ids].float()
            loss = F.cross_entropy(z, labels) / ACCUM

        pred = z.argmax(dim=1)
        loss.backward()

        total += loss.item() * ACCUM
        correct += int((pred == labels).sum().item())
        count += int(labels.numel())

        if step % ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)

        bar.set_postfix(
            train_loss=f"{total / step:.4f}",
            train_acc=f"{correct / max(1, count):.4f}",
        )

    train_loss = total / len(train_loader)
    train_acc = correct / max(1, count)
    loss, val_acc = eval_on(model, val_loader)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": loss,
        "val_acc": val_acc,
    })

    print(
        f"[atoms] epoch {epoch} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"val_loss={loss:.4f} | "
        f"val_acc={val_acc:.4f}"
    )

    if loss < best:
        best = loss
        model.save_pretrained(BEST_DIR, safe_serialization=True)
        tok.save_pretrained(BEST_DIR)
        print(f"[atoms] saved best by val_loss: {BEST_DIR}")


model.save_pretrained(LAST_DIR, safe_serialization=True)
tok.save_pretrained(LAST_DIR)

summary = {
    "model_id": MODEL,
    "objective": "cross_entropy_over_two_candidate_logits",
    "candidates": {
        "false": " False",
        "true": " True",
        "false_token_id": int(false_id),
        "true_token_id": int(true_id),
    },
    "all_path": ALL_PATH,
    "train_split_path": TRAIN_PATH,
    "val_split_path": VAL_PATH,
    "num_total_rows": len(rows),
    "num_train_rows": len(train_rows),
    "num_val_rows": len(val_rows),
    "uid_overlap_train_val": len(overlap),
    "epochs": EPOCHS,
    "train_bs": TRAIN_BS,
    "eval_bs": EVAL_BS,
    "grad_accum": ACCUM,
    "lr": LR,
    "max_length": MAX_LEN,
    "weight_decay": WD,
    "warmup_ratio": WARMUP,
    "val_fraction": VAL_FRAC,
    "lora_r": R,
    "lora_alpha": ALPHA,
    "lora_dropout": DROPOUT,
    "lora_target_modules": MODULES,
    "best_val_loss": best,
    "adapter_best_dir": BEST_DIR,
    "adapter_last_dir": LAST_DIR,
    "selection_rule": "best adapter is selected by lowest validation two-candidate probability loss",
    "history": history,
}

save_json(os.path.join(RES, "python_atoms_train_summary.json"), summary)

del model
clear()

print("\nSaved atoms adapters:")
print("best:", BEST_DIR)
print("last: ", LAST_DIR)
print("summary:", os.path.join(RES, "python_atoms_train_summary.json"))
